# OpenAI Agents SDK — Your First Agent

This practical notebook introduces the **OpenAI Agents SDK** and implements a simple agent from scratch.

## Learning Objectives

By the end of this notebook, you will understand:

- What the OpenAI Agents SDK is
- The relationship between the Agents SDK and the Responses API
- The six core concepts: Agents, Runner, Tools, Handoffs, Guardrails, and Tracing
- How to create your first agent
- How `Runner.run_sync()` executes an agent
- How asynchronous execution works with `Runner.run()`
- How the agent loop works internally
- How to retrieve the final agent response
- How the runner manages tool calls and iterative reasoning


## 1. What Is the OpenAI Agents SDK?

The **OpenAI Agents SDK** is a Python framework for building AI agents and multi-agent applications.

At a high level, an agent combines:

**LLM + Instructions + Tools + Agent Loop**

- **LLM** → the brain that understands and reasons
- **Instructions** → define the agent's role and behavior
- **Tools** → allow the agent to take actions
- **Agent loop** → repeatedly reasons, calls tools, observes results, and continues until it can provide a final answer

The Agents SDK is built on top of the OpenAI Responses API. This means the SDK provides a higher-level framework while the Responses API remains the underlying model interaction layer.

## 2. The Six Core Concepts

### 1. Agent
The agent defines the model, instructions, and optional tools that determine how the system behaves.

### 2. Runner
The runner executes the agent. It manages the agentic loop, including model calls, tool execution, and deciding when the process is complete.

### 3. Tools
Tools are functions that the agent can call to perform actions or retrieve information. Python functions can be exposed as tools.

### 4. Handoffs
Handoffs allow one agent to transfer control to another specialized agent. This is useful for multi-agent systems.

### 5. Guardrails
Guardrails validate inputs and outputs and help keep agents safe, reliable, and within the intended scope.

### 6. Tracing
Tracing provides visibility into the agent's execution. It helps you understand the steps taken, tool calls made, and overall behavior for debugging and optimization.

## 3. Installing the Agents SDK

Install the package using pip:


In [ ]:
# Run this command in a terminal or notebook cell
%pip install openai-agents

## 4. Configure Your OpenAI API Key

The SDK needs an OpenAI API key to communicate with OpenAI models.

For security, store the key in an environment variable instead of hardcoding it in your notebook.

### Windows PowerShell
```powershell
$env:OPENAI_API_KEY="your-api-key"
```

### macOS / Linux
```bash
export OPENAI_API_KEY="your-api-key"
```

Restart the notebook kernel after setting the environment variable if necessary.

In [ ]:
import os

if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY is configured.")
else:
    print("OPENAI_API_KEY is not configured. Set it before running the agent.")

## 5. Import the Agent and Runner

The first step is importing two core components:

- `Agent` → the blueprint or definition of the agent
- `Runner` → the execution engine that runs the agent


In [ ]:
from agents import Agent, Runner

## 6. Create Your First Agent

An agent needs a name and instructions. You can also specify a model.

Think of these two fields as:

- **Name** → the agent's job title
- **Instructions** → the agent's job description


In [ ]:
agent = Agent(
    name="Learning Assistant",
    instructions=(
        "You are a helpful learning assistant. "
        "Explain technical concepts clearly and use simple examples."
    ),
    model="gpt-5.5",
)

print(agent.name)

## 7. Run the Agent Synchronously

The simplest way to execute an agent is with `Runner.run_sync()`.

The runner takes care of the execution loop behind the scenes.

Conceptually, the flow is:

```text
Your Python Code
       |
       v
Runner.run_sync()
       |
       v
Agent Loop
       |
       v
Responses API
       |
       v
OpenAI Model
       |
       v
Final Answer
```


In [ ]:
result = Runner.run_sync(
    agent,
    "Explain what an AI agent is in simple terms."
)

print(result.final_output)

## 8. Understanding `result.final_output`

The runner returns a result object. The `final_output` property contains the final text produced by the agent.

This gives us a simple pattern:

```python
result = Runner.run_sync(agent, "Your question")
print(result.final_output)
```

This is the basic three-step workflow:

1. Define the agent
2. Run the agent
3. Read the final output

## 9. The Agent Loop

The important part of the Agents SDK is the runner. It manages the agentic loop.

A simplified version looks like this:

```text
User Input
    |
    v
Runner
    |
    v
Call Model
    |
    v
Does the model need a tool?
    |
    +---- No ----> Final Answer
    |
    +---- Yes ---> Execute Tool
                    |
                    v
                 Tool Result
                    |
                    v
                 Call Model Again
                    |
                    v
              Repeat Until Done
```

If no tool is needed, the model returns a final response immediately.

If a tool is needed, the runner executes it, sends the result back into the agent loop, and continues until the model produces a final answer.

## 10. How the Agents SDK Uses the Responses API

The Agents SDK is a higher-level framework built on top of the Responses API.

The simplified architecture is:

```text
Your Application
       |
       v
OpenAI Agents SDK
       |
       v
Runner / Agent Loop
       |
       v
Responses API
       |
       v
OpenAI Model
```

This means you do not have to manually implement the complete agent loop for common use cases. The runner manages the repeated model and tool interactions.

## 11. Synchronous vs Asynchronous Execution

The Agents SDK supports both synchronous and asynchronous execution.

### Synchronous
Use `Runner.run_sync()` when you want straightforward blocking execution.

```python
result = Runner.run_sync(agent, "Hello")
```

### Asynchronous
Use `Runner.run()` with `await` when building asynchronous applications.

```python
result = await Runner.run(agent, "Hello")
```

Async execution is especially useful in web applications and systems that handle multiple concurrent requests.

In [ ]:
import asyncio

async def run_agent_async():
    result = await Runner.run(
        agent,
        "Give me three practical examples of AI agents."
    )
    return result.final_output

# In a normal Python script:
# print(asyncio.run(run_agent_async()))

# In Jupyter, you can usually run:
async_result = await run_agent_async()
print(async_result)

## 12. Adding a Tool

Agents become more powerful when they can take actions using tools.

A tool is simply a function that the agent can invoke when it needs additional information or needs to perform an action.

The conceptual flow is:

```text
User asks a question
        |
        v
      Agent
        |
        v
  Model decides:
  "I need a tool"
        |
        v
    Tool Call
        |
        v
   Python Function
        |
        v
   Tool Result
        |
        v
      Agent Loop
        |
        v
   Final Response
```

The runner coordinates this process.

In [ ]:
from agents import function_tool

@function_tool
def get_course_status() -> str:
    """Return the current status of a programming course."""
    return "The course is currently active and progressing as planned."

tool_agent = Agent(
    name="Course Assistant",
    instructions=(
        "You help users with course questions. "
        "Use the course status tool when the user asks about course status."
    ),
    model="gpt-5.5",
    tools=[get_course_status],
)

tool_result = Runner.run_sync(
    tool_agent,
    "What is the current status of the course?"
)

print(tool_result.final_output)

## 13. What Happens When a Tool Is Called?

The runner handles the process approximately like this:

1. The user sends a request.
2. The runner sends the request to the model.
3. The model determines whether it needs a tool.
4. If no tool is required, the model produces the final response.
5. If a tool is required, the model generates a tool call.
6. The runner executes the Python function.
7. The tool result is returned to the agent loop.
8. The model receives the result and reasons again.
9. The loop repeats if another tool is needed.
10. The final answer is returned through `result.final_output`.

This is why the runner is such an important part of the Agents SDK.

## 14. Handoffs: Moving Between Specialized Agents

A handoff allows one agent to transfer control to another agent.

For example, imagine a customer support system:

```text
                    Customer Request
                           |
                           v
                    Triage Agent
                     /    |    \
                    /     |     \
                   v      v      v
             Billing  Technical  Sales
               Agent     Agent    Agent
```

The triage agent determines which specialist should handle the request and hands control to that agent.

Handoffs are particularly useful when each agent has a narrow responsibility.

## 15. Guardrails

Guardrails help validate what enters and leaves an agent system.

Examples include:

- Rejecting unsafe input
- Ensuring the agent stays on topic
- Validating structured outputs
- Preventing sensitive information from being returned
- Enforcing business rules

A useful mental model is:

```text
User Input
    |
    v
Input Guardrail
    |
    v
   Agent
    |
    v
Output Guardrail
    |
    v
Final Response
```

Guardrails become increasingly important as agents move from experiments into production.

## 16. Tracing and Debugging

Agent systems can involve many steps. A user might see only one final response, while internally the system may have:

- Called the model multiple times
- Invoked several tools
- Passed tool results back into the model
- Performed handoffs
- Applied guardrails

Tracing helps you understand this execution history.

Conceptually:

```text
User Request
    |
    +--> Model Call
    |
    +--> Tool Call
    |
    +--> Tool Result
    |
    +--> Model Call
    |
    +--> Final Response
```

Tracing is valuable when debugging unexpected behavior, measuring performance, or understanding why an agent made a particular decision.

## 17. Agents SDK vs Building the Loop Yourself

Without a higher-level agent framework, you may need to manually manage:

- Model requests
- Tool schemas
- Tool calls
- Tool execution
- Tool results
- Repeated model calls
- Stop conditions
- Errors and retries

With the Agents SDK, the runner handles much of the common orchestration.

| Responsibility | Manual Approach | Agents SDK |
|---|---|---|
| Define agent | You manage | `Agent` |
| Run loop | You manage | `Runner` |
| Tool integration | You manage | SDK tools |
| Multi-agent routing | Custom logic | Handoffs |
| Validation | Custom logic | Guardrails |
| Debugging | Custom logging | Tracing |

The key benefit is not that the model becomes smarter. The benefit is that the surrounding agent orchestration becomes easier to build and maintain.

## 18. A Complete Minimal Example

The core pattern can be summarized in a very small amount of code:


In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Simple Assistant",
    instructions="You are a helpful assistant who explains concepts clearly.",
    model="gpt-5.5",
)

result = Runner.run_sync(
    agent,
    "Explain the difference between an AI model and an AI agent."
)

print(result.final_output)

## 19. Practice Exercise 1 — Create a Custom Agent

Create an agent called `Python Tutor`.

Requirements:

1. The agent should help beginners learn Python.
2. It should explain concepts in simple language.
3. It should include a short example when appropriate.
4. Run it with a question such as: `What is a Python list?`

In [ ]:
# TODO: Create your Python Tutor agent here

# python_tutor = Agent(
#     name="Python Tutor",
#     instructions="...",
#     model="gpt-5.5",
# )

# result = Runner.run_sync(
#     python_tutor,
#     "What is a Python list?"
# )

# print(result.final_output)

## 20. Practice Exercise 2 — Add a Tool

Create a tool that returns a fixed weather message, for example:

`The current weather is sunny with a temperature of 28°C.`

Then create an agent that uses this tool when asked about the weather.

The objective is to understand the flow:

```text
User Question
     |
     v
Agent
     |
     v
Model Requests Tool
     |
     v
Weather Function
     |
     v
Tool Result
     |
     v
Final Answer
```

## 21. Practice Exercise 3 — Think Like a Multi-Agent Architect

Design a customer support system with three specialized agents:

- **Triage Agent** → identifies the customer's issue
- **Billing Agent** → handles payment and invoice questions
- **Technical Agent** → handles technical problems

Answer these questions:

1. Which agent should receive the first message?
2. When should a handoff happen?
3. What guardrails would you add?
4. What information would you want in tracing logs?


## 22. Key Takeaways

### The OpenAI Agents SDK provides a higher-level way to build agents.

The six concepts to remember are:

1. **Agent** → defines the agent's identity, instructions, model, and tools.
2. **Runner** → executes and manages the agent loop.
3. **Tools** → allow the agent to interact with external systems and perform actions.
4. **Handoffs** → allow control to move between specialized agents.
5. **Guardrails** → validate and constrain agent behavior.
6. **Tracing** → provides visibility into agent execution.

The most important architectural idea is the **Runner**. It acts as the execution engine that coordinates model calls, tool calls, tool results, and the loop that continues until a final response is produced.

The basic mental model is:

**Agent definition → Runner execution → Agent loop → Final output**

Once you understand this foundation, the next logical topics are tools, handoffs, guardrails, and tracing.